<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 17. ViT — 이미지를 patch token으로 — An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale

- **원 논문:** [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929)
- **저자 / 발표:** Alexey Dosovitskiy et al. · ICLR (2021)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** convolution shape, self-attention, class token, positional embedding

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

식 (1)의 patchify·linear projection·class token·position embedding과 식 (2)~(4)의 pre-LN Transformer encoder를 8×8 영상에 구현합니다. 위치 정보가 없으면 patch 순열을 구별하지 못하는 성질과 작은 막대 분류 학습을 확인합니다.

**원 논문과 다른 것**

JFT-300M/ImageNet 사전학습과 16×16 patch의 ViT-Base/Large/Huge는 생략합니다. 8×8 합성 영상, 2×2 patch, encoder 1층을 사용하므로 표 1~6의 전이 성능은 검증하지 않습니다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1 식 (1), 그림 1 | patchify·E projection·[class]·E_pos | [B,N+1,D] shape |
| §3.1 식 (2) | pre-LN multi-head attention residual | token shape 보존 |
| §3.1 식 (3) | pre-LN MLP residual | 유한 gradient |
| §3.1 식 (4) | class token 분류 head | 두 막대 class 정확도 |
| §3.1 Inductive bias | position embedding 제거 ablation | patch 순열 불변성 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
z_0=[x_{class};x_p^1E;\dots;x_p^NE]+E_{pos},\qquad
N=\frac{HW}{P^2}
$$

- $x_p^i$는 $P\times P$ image patch, $E$는 patch projection, $E_{pos}$는 position embedding입니다.
- 2D image를 token sequence로 바꾸어 Transformer encoder에 입력하는 것이 핵심입니다.
- patchify, linear projection, class token, position embedding, encoder, head를 별도 모듈로 보입니다.
- image $[B,C,H,W]$는 patch $[B,N,CP^2]$를 거쳐 token $[B,N+1,D]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1 식 (1), 그림 1
- **노트북 구현:** patchify·E projection·[class]·E_pos
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** [B,N+1,D] shape를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** JFT-300M/ImageNet 사전학습과 16×16 patch의 ViT-Base/Large/Huge는 생략합니다. 8×8 합성 영상, 2×2 patch, encoder 1층을 사용하므로 표 1~6의 전이 성능은 검증하지 않습니다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(ACCELERATOR.summary())
print("synthetic-data preprocessing stays on CPU; neural training uses DEVICE")

## 1) 식 (1)의 patch sequence

8×8 이미지를 겹치지 않는 2×2 patch로 나누면 `N=(8/2)^2=16`, patch vector 차원은
`P²C=4`입니다. `patchify`와 `unpatchify`가 정확한 역함수인지 먼저 확인합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1 식 (1), 그림 1
- **이 셀의 책임:** patchify·E projection·[class]·E_pos
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** [B,N+1,D] shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 17-1: [B,C,H,W] ↔ [B,N,P²C] 변환을 구현하세요.
def patchify(x: Tensor, patch_size: int) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def unpatchify(tokens: Tensor, patch_size: int, height: int, width: int) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 2) 식 (1)~(4)의 TinyViT

원 논문은 각 encoder block을 `x' = MSA(LN(x)) + x`,
`x = MLP(LN(x')) + x'`로 씁니다. 여기서는 이 pre-LN 순서를 고수준
`TransformerEncoder` 호출에 숨기지 않습니다. `ViTSelfAttention`은 Q/K/V와 head
결합을, `ViTMLP`는 GELU를 포함한 식 (3)을, `ViTEncoderBlock`은 두 residual
경로를 각각 드러냅니다.

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.1 식 (1), 그림 1 / §3.1 식 (2) / §3.1 식 (3) / §3.1 식 (4)
- **이 셀의 책임:** patchify·E projection·[class]·E_pos / pre-LN multi-head attention residual / pre-LN MLP residual / class token 분류 head
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** [B,N+1,D] shape / token shape 보존 / 유한 gradient / 두 막대 class 정확도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 17-2: MSA, MLP, pre-LN residual block을 직접 구현하세요.
# TODO 17-3: patch projection, class/position token, encoder를 조립하세요.
class ViTSelfAttention(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, dim: int, heads: int):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def _split_heads(self, tensor: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def _merge_heads(self, tensor: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, hidden: Tensor) -> tuple[Tensor, Tensor]:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class ViTMLP(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, dim: int, hidden_dim: int):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class ViTEncoderBlock(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, dim: int, heads: int, mlp_dim: int):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def forward(self, hidden: Tensor) -> tuple[Tensor, Tensor]:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


class TinyViT(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, image_size=8, patch_size=2, dim=24, heads=3, classes=2):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def embed_tokens(self, x: Tensor, use_position: bool = True) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def encode_tokens(self, tokens: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def forward(self, x: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 논문 주장 확인 실험

- **원문 위치:** §3.1 Inductive bias
- **이 셀의 책임:** position embedding 제거 ablation
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** patch 순열 불변성. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 17-4: patch 순서를 섞어 position 유무의 class representation을 비교하세요.
tokens_no_pos = vit.embed_tokens(dummy, use_position=False)
permutation = torch.randperm(16)
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §3.1 식 (4)
- **이 셀의 책임:** class token 분류 head
- **Tensor shape 계약:** image [B,C,H,W] → patches [B,N,CP²] → tokens [B,N+1,D]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 두 막대 class 정확도. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 17-5: 세로/가로 막대 합성 데이터를 만들고 class token head를 학습하세요.
_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.